https://youtu.be/4snyQbdhTwM?si=38MhvsI-dwpaAQhU

In [0]:
source_data = [(1,'A'),(2,'B'),(3,'C'),(4,'D'),(5,'E')]
source_schema = ['id','name']
source_df = spark.createDataFrame(source_data,source_schema)
source_df.show()

target_data = [(1,'A'),(2,'B'),(3,'X'),(4,'F'),(6,'G')]
target_schema = ['id','name']
target_df = spark.createDataFrame(target_data,target_schema)
target_df.show()

+---+----+
| id|name|
+---+----+
|  1|   A|
|  2|   B|
|  3|   C|
|  4|   D|
|  5|   E|
+---+----+

+---+----+
| id|name|
+---+----+
|  1|   A|
|  2|   B|
|  3|   X|
|  4|   F|
|  6|   G|
+---+----+



In [0]:
from pyspark.sql.functions import *

In [0]:
df = (
    source_df
    .join(target_df,source_df.id == target_df.id,'full')
    .withColumn("status",
                when((source_df.id == target_df.id) & (source_df.name == target_df.name),"Matched")
                .when((source_df.id == target_df.id) & (source_df.name != target_df.name),"Un-Matched")
                .when((source_df.id.isNotNull() & target_df.id.isNull()), "Target Missing")
                .when((source_df.id.isNull() & target_df.id.isNotNull()), "Source Missing")
                )
    .select(
        source_df.id.alias("source_id"),
        target_df.id.alias("target_id"),
        "status"

        )
)
df.display()

source_id,target_id,status
1,1,Matched
2,2,Matched
3,3,Un-Matched
4,4,Un-Matched
5,null,Target Missing
null,6,Source Missing
